In [1]:
import torch
import torch.nn as nn

c:\Users\Manoj\Python ML\Lib\site-packages\torch\cuda\__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
import torch
import torch.nn as nn

class RoPE(nn.Module):
    def __init__(self, dim, base=10000):
        """
        dim: head dimension (must be even)
        base: frequency base (default 10000)
        """
        super().__init__()
        assert dim % 2 == 0, "Dimension must be even for RoPE"

        self.dim = dim
        self.base = base

        # Precompute inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        """
        x: (batch, seq_len, dim)
        """
        seq_len = x.shape[1]

        # Positions
        positions = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)

        # Compute angles: (seq_len, dim/2)
        angles = torch.einsum("i,j->ij", positions, self.inv_freq)

        # Create sin and cos: (seq_len, dim)
        sin = torch.sin(angles)
        cos = torch.cos(angles)

        # Expand to match x: (1, seq_len, dim/2)
        sin = sin.unsqueeze(0)
        cos = cos.unsqueeze(0)

        # Split x into even and odd parts
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        # Apply rotation
        x_rotated = torch.stack([
            x1 * cos - x2 * sin,
            x1 * sin + x2 * cos
        ], dim=-1)

        # Reshape back to original dim
        x_rotated = x_rotated.flatten(-2)

        return x_rotated